# CHIIR 2026 Tutorial on Controlled Experimentation of Model Search Behaviour with Geniie-Lab

## Setup geniie-lab

## Preparation

Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Checkout and install geniie-analytics (1st time only)

In [ ]:
!git clone https://github.com/geniie-lab/geniie-analytics.git -b dev /content/drive/MyDrive/geniie-analytics

In [ ]:
!cd /content/drive/MyDrive/geniie-analytics && python -m pip install -r requirements_colab.txt

In [ ]:
!cd /content/drive/MyDrive/geniie-analytics && python -m pip install -e .

♻ Restart a runtime and continue below ⬇

Upload or copy log files to `/drive/MyDrive/geniie-analytics/logs` folder using the file panel on the left

Import libraries

In [ ]:
# Standard libraries
from pathlib import Path

# Local application imports
from geniie_analytics.io import read_logs
from geniie_analytics.summary import summarise
from geniie_analytics.query import query_length, vocab_size
from geniie_analytics.ranking import retrieval_performance
from geniie_analytics.click import click_size, click_position, position_plot
from geniie_analytics.relevance import relevance_judgement, confusion_matrix
from geniie_analytics.qrels import apply_qrels
from geniie_analytics.sctr import compute_sctr
from geniie_analytics.descriptive import descriptive_table, descriptive_plot
from geniie_analytics.anova import run_anova, anova_to_latex

## Load log data

In [ ]:
# Read log files (*.jsonl)
log_dir = Path("/content/drive/MyDrive/geniie-analytics/logs") # Change here if needed
log_files = log_dir.glob("qwen3_*_recall.jsonl")
df = read_logs(log_files)

# Optional name mapping for better readability
df["model"] = df["model"].replace({
    "qwen/qwen3-235b-a22b-2507": "qwen3 llm",
    "qwen/qwen3-vl-235b-a22b-instruct": "qwen3 vlm",
    "openai/gpt-5-mini": "gpt5 llm",
    "openai/gpt-5-image-mini": "gpt5 vlm"
})

# inspect
display(df.head())

## Summary

In [ ]:
summary = summarise(df, exclude=["topic_id", "query", "docid", "doc_ids", "docids", "rankings", "performance", "qrel_labels", "created_at", "reason"])
for col, stats in summary.items():
    display(stats)

## Query

### Query Length

In [ ]:
qlen_df = query_length(df)
display(qlen_df.head())

In [ ]:
table = descriptive_table(qlen_df, dv="query_length", ivs=["model"])
display(table)

In [ ]:
descriptive_plot(table, dv="query_length", ivs=["model"])

In [ ]:
anova_table = run_anova(qlen_df, dv="query_length", ivs=["model"])
display(anova_table)

In [ ]:
# Pro tip: Export ANOVA table to LaTeX
latex_table = anova_to_latex(
    anova_table,
    caption="ANOVA table for query length by models",
    label="tab:anova_qlen",
    float_format="%.3f"
)
print(latex_table)

### Vocabulary Size

In [ ]:
vocab_df = vocab_size(df)
display(vocab_df.head())

In [ ]:
table = descriptive_table(vocab_df, dv="vocab_size", ivs=["model"])
display(table)

In [ ]:
descriptive_plot(table, dv="vocab_size", ivs=["model"])

In [ ]:
anova_table = run_anova(vocab_df, dv="vocab_size", ivs=["model"])
display(anova_table)

### 🦾Exercise

Summarise your findings on query formulation

## Ranking

In [ ]:
# Measure retrieval performance
performance_df = retrieval_performance(df)
display(performance_df.head())

In [ ]:
table = descriptive_table(performance_df, dv="nDCG@10", ivs=["model"])
display(table)

In [ ]:
descriptive_plot(table, dv="nDCG@10", ivs=["model"])

In [ ]:
# topic by topic comparison at iteration 0
table = descriptive_table(performance_df, dv="nDCG@10", ivs=["model", "topic_id"])
descriptive_plot(table, dv="nDCG@10", ivs=["model", "topic_id"], bar_width=0.2)

### 🦾 Exercise

Repeat the same analysis for `RR@10` metric

## Click

### Click Size

In [ ]:
click_size_df = click_size(df)
display(click_size_df.head())

In [ ]:
table = descriptive_table(click_size_df, dv="click_size", ivs=["model"])
display(table)

In [ ]:
descriptive_plot(table, dv="click_size", ivs=["model"])

In [ ]:
anova_table = run_anova(click_size_df, dv="click_size", ivs=["model"])
display(anova_table)

### Click Position

In [ ]:
click_position_df = click_position(df)
display(click_position_df.head())

In [ ]:
position_plot(click_position_df)

### 🦾Exercise

What can you say about LLM's click pattern when compared to human searchers?

### SCTR: Successful Click-through Rate

In [ ]:
# Add qrel labels to ranking
df = apply_qrels(df)
# Calculate Successful Click-through Rate (SCTR)
sctr = compute_sctr(df)
# inspect
display(sctr.head())

In [ ]:
table = descriptive_table(sctr, dv="f1", ivs=["model"])
display(table)

In [ ]:
descriptive_plot(table, dv="f1", ivs=["model"])

In [ ]:
anova_table = run_anova(sctr, dv="f1", ivs=["model"])
display(anova_table)

## Relevance Judgements

In [ ]:
rel_judge_df = relevance_judgement(df)
display(rel_judge_df.head())

In [ ]:
table = descriptive_table(rel_judge_df, dv="correct", ivs=["model"])
display(table)

In [ ]:
descriptive_plot(table, dv="accuracy", ivs=["model"])

### Confusion Matrix

In [ ]:
cms = confusion_matrix(rel_judge_df)

for model, cm in cms.items():
    print(f"Model: {model}")
    display(cm)

### 🦾Exercise

LLMs are known to generate positive responses. Where did you observe such a pattern in the confusion matrix?

## What's Next?

You should have the following set of log files. Try to make new comparisons to deepen our understanding of model search behaviour!

* Qwen 3 + LLM + High-Recall Task Instruction
* Qwen 3 + VLM + High-Recall Task Instruction
* 🆕 Qwen 3 + LLM + **High-Precision** Task Instruction
* 🆕 Qwen 3 + VLM + **High-Precision** Task Instruction
* 🆕 **gpt5 mini** + LLM + High-Recall Task Instruction
* 🆕 **gpt5 mini** + VLM + High-Recall Task Instruction

Don't forget to change the following matching rule to select relevant files for comparison

```
log_files = log_dir.glob("qwen3_*_recall.jsonl")
```